# Task 1 — Fine-tuned vs. No-fine-tune Baseline

Compares the **fine-tuned** Llama-3.1-8B QLoRA run against the **un-fine-tuned base model** on Task 1 (Risk Clause Recognition, binary `Yes`/`No`), evaluated on the *same* validation set.

Inputs (both produced by the Kaggle runs and downloaded via `kaggle/run.ps1`):

| Run | Notebook that produced it | Artifacts read here |
|---|---|---|
| Fine-tuned | `llm_fine_tuning_LORA_task1_v2.ipynb` | `kaggle_output_task1_fine_tuned/eval_metrics.json` |
| Baseline (no fine-tune) | `llama_3.1_task_1_no_fine_tune.ipynb` | `kaggle_output_task1_baseline/no_finetune_baseline/eval_metrics.json` |

Both notebooks build the validation set with the identical pipeline (`random.seed(42)`, contract-level split), so the comparison is apples-to-apples — and a cell below verifies that from the saved JSONL files.

Structure:
1. **Setup & loaders** — paths, palette, load both metric files, sanity checks.
2. **Headline comparison** — summary table with deltas.
3. **Per-class breakdown** — precision / recall / F1, baseline vs. fine-tuned.
4. **Confusion matrices & error analysis** — where each model goes wrong.
5. **Takeaways** — computed from the numbers, so they stay correct if you point at other runs.

## 1. Setup & loaders

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ---- Configuration: point these at any two runs with the standard layout ----
# Each run dir holds `eval_metrics.json` + the `cuad/validation/` JSONL it was scored on.
FINETUNED_DIR = Path("kaggle_output_task1_fine_tuned")
BASELINE_DIR = Path("kaggle_output_task1_baseline") / "no_finetune_baseline"

# ---- Palette: one fixed color per model, everywhere in this notebook ----
# (validated pair: CVD dE 73.6; the aqua is <3:1 on white, so every bar carries
#  a visible value label rather than relying on color alone)
BASE_COLOR = "#1baf7a"   # baseline (aqua)
FT_COLOR   = "#2a78d6"   # fine-tuned (blue)
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

# One-hue sequential ramp for the confusion-matrix heatmaps.
SEQ_BLUES = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#9ec5f4", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"])

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.labelcolor": INK, "text.color": INK,
})

for label, d in [("Fine-tuned", FINETUNED_DIR), ("Baseline", BASELINE_DIR)]:
    assert (d / "eval_metrics.json").exists(), (
        f"{label} run has no eval_metrics.json at: {(d / 'eval_metrics.json').resolve()}")
print("Fine-tuned run:", FINETUNED_DIR.resolve())
print("Baseline run:  ", BASELINE_DIR.resolve())

In [ ]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def bar_labels(ax, fmt="{:.2f}"):
    """Annotate each bar with its height, in text ink (never the series color)."""
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2, h),
                    ha="center", va="bottom", fontsize=9, color=INK)


base_metrics = load_json(BASELINE_DIR / "eval_metrics.json")
ft_metrics = load_json(FINETUNED_DIR / "eval_metrics.json")

# Fixed display order: baseline first, fine-tuned second — reused by every chart.
RUNS = {
    "Baseline (no fine-tune)": base_metrics,
    "Fine-tuned": ft_metrics,
}
RUN_COLORS = {"Baseline (no fine-tune)": BASE_COLOR, "Fine-tuned": FT_COLOR}
CLASSES = ["Yes", "No"]

# Sanity: both runs must have been evaluated on the same validation set.
assert base_metrics["n_validation_examples"] == ft_metrics["n_validation_examples"], (
    f"Different validation sizes: baseline={base_metrics['n_validation_examples']} "
    f"vs fine-tuned={ft_metrics['n_validation_examples']} — the runs are NOT comparable.")
for cls in CLASSES:
    b = base_metrics["classification_report"][cls]["support"]
    f = ft_metrics["classification_report"][cls]["support"]
    assert b == f, f"Class '{cls}' support differs: baseline={b} vs fine-tuned={f}"

for name, m in RUNS.items():
    print(f"{name:>24}: {m['model_name']}  "
          f"({m['n_validation_examples']} validation examples)")

In [ ]:
# Verify the two runs saw the SAME validation examples, not just the same count.
# Each run saves the JSONL it evaluated on; compare them record-by-record
# (order-insensitive, in case of incidental row-order differences).
def load_val_records(run_dir):
    path = run_dir / "cuad" / "validation" / "cuad_validation.jsonl"
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return sorted(json.dumps(json.loads(line), sort_keys=True)
                      for line in f if line.strip())


base_val = load_val_records(BASELINE_DIR)
ft_val = load_val_records(FINETUNED_DIR)

if base_val is None or ft_val is None:
    print("WARNING: a validation JSONL is missing; falling back on the "
          "size/support checks above.")
elif base_val == ft_val:
    print(f"OK — both runs evaluated the identical {len(base_val)} validation examples.")
else:
    overlap = len(set(base_val) & set(ft_val))
    raise AssertionError(
        f"Validation sets DIFFER (only {overlap}/{len(base_val)} examples shared) — "
        "the metric comparison below would not be apples-to-apples.")

## 2. Headline comparison

Accuracy alone is misleading under class imbalance (`No` is ~3/4 of the validation set, so "always No" scores ~75%). The table therefore leads with **macro F1** and the per-class F1 scores; the `Δ` row is fine-tuned minus baseline.

In [ ]:
def headline(m):
    r = m["classification_report"]
    return {
        "accuracy": m["accuracy"],
        "macro F1": r["macro avg"]["f1-score"],
        "Yes F1": r["Yes"]["f1-score"],
        "No F1": r["No"]["f1-score"],
        "Yes precision": r["Yes"]["precision"],
        "Yes recall": r["Yes"]["recall"],
    }


summary = pd.DataFrame({name: headline(m) for name, m in RUNS.items()}).T
summary.loc["\u0394 (fine-tuned \u2212 baseline)"] = (
    summary.loc["Fine-tuned"] - summary.loc["Baseline (no fine-tune)"])
summary.style.format("{:+.1%}", subset=(["\u0394 (fine-tuned \u2212 baseline)"], slice(None))) \
    .format("{:.1%}", subset=(list(RUNS), slice(None)))

## 3. Per-class breakdown

Precision / recall / F1 for each class, baseline next to fine-tuned. The `Yes` panel is the one that matters most for Task 1 — `Yes` (risk clause present) is the rare class the project cares about catching.

In [ ]:
def per_class_comparison(runs, classes=CLASSES,
                          metrics=("precision", "recall", "f1-score")):
    """One panel per class; within each panel, baseline vs fine-tuned per metric."""
    fig, axes = plt.subplots(1, len(classes), figsize=(11, 4), sharey=True)
    for ax, cls in zip(axes, classes):
        data = pd.DataFrame(
            {name: [m["classification_report"][cls][x] for x in metrics]
             for name, m in runs.items()},
            index=[x.replace("f1-score", "F1") for x in metrics],
        )
        data.plot.bar(ax=ax, color=[RUN_COLORS[n] for n in data.columns],
                      width=0.75, legend=False)
        support = int(runs[list(runs)[0]]["classification_report"][cls]["support"])
        ax.set_title(f'class "{cls}"  (n={support})')
        ax.set_ylim(0, 1.12)
        ax.tick_params(axis="x", rotation=0)
        bar_labels(ax)
    axes[0].set_ylabel("score")
    axes[0].legend(loc="upper left", frameon=False)
    fig.suptitle("Per-class performance on the shared validation set", y=1.02)
    plt.tight_layout()
    plt.show()


per_class_comparison(RUNS)

## 4. Confusion matrices & error analysis

Same color scale on both heatmaps, so darkness is directly comparable between models.

In [ ]:
def confusion_side_by_side(runs):
    """Both confusion matrices on one shared color scale."""
    matrices = {name: pd.DataFrame(m["confusion_matrix"]["matrix"],
                                   index=m["confusion_matrix"]["labels"],
                                   columns=m["confusion_matrix"]["labels"])
                for name, m in runs.items()}
    vmax = max(mat.values.max() for mat in matrices.values())

    fig, axes = plt.subplots(1, len(matrices), figsize=(9.5, 4))
    for ax, (name, mat) in zip(axes, matrices.items()):
        labels = list(mat.index)
        ax.imshow(mat, cmap=SEQ_BLUES, vmin=0, vmax=vmax)
        ax.set_xticks(range(len(labels)), labels)
        ax.set_yticks(range(len(labels)), labels)
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
        ax.set_title(name)
        ax.grid(False)
        for i in range(len(labels)):
            for j in range(len(labels)):
                v = mat.iloc[i, j]
                ax.text(j, i, str(v), ha="center", va="center", fontsize=12,
                        color="white" if v > vmax / 2 else INK)
    plt.tight_layout()
    plt.show()


confusion_side_by_side(RUNS)

In [ ]:
def error_counts(m):
    """Split each model's mistakes into the two error types."""
    cm = m["confusion_matrix"]
    mat = pd.DataFrame(cm["matrix"], index=cm["labels"], columns=cm["labels"])
    return {
        "missed clause (Yes\u2192No)": int(mat.loc["Yes", "No"]),
        "false alarm (No\u2192Yes)": int(mat.loc["No", "Yes"]),
    }


errors = pd.DataFrame({name: error_counts(m) for name, m in RUNS.items()})

ax = errors.plot.bar(color=[RUN_COLORS[n] for n in errors.columns], width=0.7)
ax.set_title("Errors by type (lower is better)")
ax.set_ylabel("# validation examples")
ax.tick_params(axis="x", rotation=0)
ax.legend(frameon=False)
bar_labels(ax, fmt="{:.0f}")
plt.tight_layout()
plt.show()

total_base = errors["Baseline (no fine-tune)"].sum()
total_ft = errors["Fine-tuned"].sum()
n = ft_metrics["n_validation_examples"]
print(f"Total errors — baseline: {total_base}/{n}, fine-tuned: {total_ft}/{n}")
print(f"Fine-tuning removed {total_base - total_ft} errors "
      f"({(total_base - total_ft) / total_base:.0%} error reduction).")

## 5. Takeaways

Computed from the loaded metrics, so this stays correct when `FINETUNED_DIR` / `BASELINE_DIR` point at different runs.

In [ ]:
b, f = base_metrics["classification_report"], ft_metrics["classification_report"]

print(f"Validation set: {ft_metrics['n_validation_examples']} examples "
      f"({int(f['Yes']['support'])} Yes / {int(f['No']['support'])} No)\n")
print(f"1. Accuracy:  {base_metrics['accuracy']:.1%} -> {ft_metrics['accuracy']:.1%} "
      f"({ft_metrics['accuracy'] - base_metrics['accuracy']:+.1%})")
print(f"2. Macro F1:  {b['macro avg']['f1-score']:.1%} -> {f['macro avg']['f1-score']:.1%} "
      f"({f['macro avg']['f1-score'] - b['macro avg']['f1-score']:+.1%})")
print(f"3. Rare-class ('Yes') F1: {b['Yes']['f1-score']:.1%} -> {f['Yes']['f1-score']:.1%} — "
      "the metric fine-tuning is aimed at, since risk clauses are the minority class.")
print(f"4. 'Yes' precision {b['Yes']['precision']:.1%} -> {f['Yes']['precision']:.1%} "
      f"and recall {b['Yes']['recall']:.1%} -> {f['Yes']['recall']:.1%}: the base model "
      "flags clauses near-indiscriminately; the fine-tuned model actually recognizes them.")

no_recall_base = b["No"]["recall"]
if no_recall_base < 0.6:
    print(f"5. Baseline 'No' recall is only {no_recall_base:.1%} — on hard negatives "
          "(real clause text from another category) the un-tuned model says 'Yes' to "
          "roughly half of everything, confirming it cannot distinguish clause TYPES "
          "out of the box.")